# Inference - Base Model

In [1]:
# %env OPENAI_API_KEY=
# %env ANTHROPIC_API_KEY=
# Or set these environment variables in your system
from dotenv import load_dotenv
YOUR_DOTENV_PATH = "../.env"
load_dotenv(YOUR_DOTENV_PATH)

# Disable logging for the collabllm package
# Set to 1 to see the process of the reward computation.
%env ENABLE_COLLABLLM_LOGGING=0 

env: ENABLE_COLLABLLM_LOGGING=0


## Example 1: Movie Recommendation

In [2]:
import sys
sys.path.append('..')

import logging
logging.getLogger("LiteLLM").setLevel(logging.CRITICAL)

In [4]:
task_desc = "Recommend a movie."
single_turn_prompt = "Find a film that suitable for a date night. It should deliver an epic romantic drama, ideally in the 20th-century America, and carry the same decades-long, nostalgic storytelling spirit as Forrest Gump."

model_name = "meta-llama/Llama-3.2-3B-Instruct"

In [5]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load model and tokenizer
print(f"Loading model: {model_name}")
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_name, 
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    device_map="auto",
    trust_remote_code=True
)

print(f"Model loaded successfully!")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

# Set up tokenizer padding
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def generate_response(messages, max_new_tokens=512, temperature=0.8):
    """Generate response from the model for given messages
    
    Args:
        messages: List of message dicts with 'role' and 'content' keys
                 OR a single string (will be treated as user message)
    """
    # Handle backward compatibility with string input
    if isinstance(messages, str):
        messages = [{"role": "user", "content": messages}]
    
    # Apply chat template
    formatted_prompt = tokenizer.apply_chat_template(
        messages, 
        tokenize=False, 
        add_generation_prompt=True
    )
    
    # Tokenize
    inputs = tokenizer(formatted_prompt, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    
    # Generate
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    # Decode response (only the generated part)
    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    
    return response.strip()

print("Ready for inference!")


Loading model: meta-llama/Llama-3.2-3B-Instruct


config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/1.46G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Model loaded successfully!
Model parameters: 3,212,749,824
Ready for inference!


In [6]:
# Generate response using task_desc as system prompt and single_turn_prompt as user prompt
print("=" * 60)
print("GENERATING RESPONSE WITH SYSTEM + USER PROMPTS")
print("=" * 60)
print(f"System prompt: {task_desc}")
print(f"User prompt: {single_turn_prompt}")
print("\n" + "-" * 60)

# Create conversation with system and user messages
messages = [
    {"role": "system", "content": task_desc},
    {"role": "user", "content": single_turn_prompt}
]

# Apply chat template
formatted_prompt = tokenizer.apply_chat_template(
    messages, 
    tokenize=False, 
    add_generation_prompt=True
)

print("Formatted prompt:")
print(formatted_prompt)
print("\n" + "-" * 60)

# Tokenize
inputs = tokenizer(formatted_prompt, return_tensors="pt")
inputs = {k: v.to(model.device) for k, v in inputs.items()}

# Generate
print("Generating response...")
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.8,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )

# Decode response (only the generated part)
generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]
response = tokenizer.decode(generated_tokens, skip_special_tokens=True)

print("\nModel Response:")
print("=" * 60)
print(response.strip())


GENERATING RESPONSE WITH SYSTEM + USER PROMPTS
System prompt: Recommend a movie.
User prompt: Find a film that suitable for a date night. It should deliver an epic romantic drama, ideally in the 20th-century America, and carry the same decades-long, nostalgic storytelling spirit as Forrest Gump.

------------------------------------------------------------
Formatted prompt:
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 28 Aug 2025

Recommend a movie.<|eot_id|><|start_header_id|>user<|end_header_id|>

Find a film that suitable for a date night. It should deliver an epic romantic drama, ideally in the 20th-century America, and carry the same decades-long, nostalgic storytelling spirit as Forrest Gump.<|eot_id|><|start_header_id|>assistant<|end_header_id|>



------------------------------------------------------------
Generating response...

Model Response:
What a wonderful request! Based on your criteria, I'd like to recom